# UP2.4 · Estadística y probabilidad I: describir una variable

Ejemplos de la presentación, en el mismo orden. Este notebook va en la raíz del proyecto, junto a la carpeta `datos/`.

Ejecuta las celdas de una en una, de arriba abajo. Cada sección termina con una tabla de referencia.

Los datos son **simulados**: los barrios de Benidorm son reales, pero los anuncios y los precios se han generado con NumPy.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pisos: pd.DataFrame = pd.read_csv(Path("datos") / "pisos.csv")
precio: pd.Series = pisos["precio"]
metros: pd.Series = pisos["metros"]
columnas: list[str] = ["barrio", "tipo", "metros", "precio"]

print(precio.head(8).tolist())

[1830, 570, 1040, 1480, 790, 1120, 1580, 1640]


## 1. Variables y frecuencias

In [2]:
print(pisos.nunique())

id                   600
fecha                233
barrio                10
tipo                   4
metros               104
habitaciones           5
banos                  3
planta                27
ascensor               2
terraza                2
vistas_mar             2
anunciante             2
precio               194
alquilado_30_dias      2
dtype: int64


In [3]:
tipo: pd.Series = pisos["tipo"]
print(tipo.value_counts())
print(tipo.value_counts(normalize=True).round(3))
print(round(pisos["ascensor"].mean(), 3))

tipo
piso       411
estudio     95
ático       53
dúplex      41
Name: count, dtype: int64
tipo
piso       0.685
estudio    0.158
ático      0.088
dúplex     0.068
Name: proportion, dtype: float64
0.768


### Referencia: tipos de variable

| Tipo | Ejemplos en `pisos` | Cómo se describe |
|---|---|---|
| Numérica continua | `metros`, `precio` | Centro, dispersión y forma |
| Numérica discreta | `habitaciones`, `banos`, `planta` | Como las continuas; la moda también tiene sentido |
| Categórica | `barrio`, `tipo`, `anunciante` | Frecuencias: `value_counts()` |
| Binaria | `ascensor`, `terraza`, `vistas_mar` | La proporción de `True`: `mean()` |

## 2. El centro: media, mediana y moda

In [4]:
alquileres: pd.Series = pd.Series([800, 900, 950, 1000, 1100])
print(alquileres.sum() / len(alquileres))
print(alquileres.mean())
print(round(precio.mean(), 1))

950.0
950.0
1243.0


In [5]:
print(alquileres.median())

seis: pd.Series = pd.Series([800, 900, 950, 1000, 1100, 1200])
print(seis.median())

print(precio.median())
print(round((precio <= 1150).mean(), 3))

950.0
975.0
1150.0
0.503


In [6]:
atico: pd.Series = pd.Series([800, 900, 950, 1000, 1100, 4000])
print(round(atico.mean(), 1), atico.median())
print(round(precio.mean(), 1), precio.median())

1458.3 975.0
1243.0 1150.0


In [7]:
habitaciones: pd.Series = pisos["habitaciones"]
print(habitaciones.mode().iloc[0])
print(precio.mode().iloc[0], (precio == 1170).sum())

3
1170 10


In [8]:
print(pisos["planta"].count(), len(pisos))
print(round(pisos["planta"].mean(), 2))

581 600
7.97


### Referencia: medidas de centro

| Código | Qué da | Cuándo usarla |
|---|---|---|
| `s.mean()` | Media: la suma entre el número de valores; el punto de equilibrio | Variables numéricas sin valores extremos |
| `s.median()` | Mediana: el valor central con los datos ordenados | Cuando hay valores extremos o asimetría (precios, sueldos) |
| `s.mode()` | Moda: el valor más frecuente (devuelve una Series) | Categóricas y discretas |

Las tres ignoran los huecos (`NaN`): se calculan con los valores que hay. `s.count()` dice cuántos son.

## 3. La dispersión

In [9]:
estable: pd.Series = pd.Series([950, 1000, 1000, 1050])
variable: pd.Series = pd.Series([500, 800, 1200, 1500])
print(estable.mean(), variable.mean())
print(estable.max() - estable.min())
print(variable.max() - variable.min())

1000.0 1000.0
100
1000


In [10]:
desvios: pd.Series = variable - variable.mean()
print(desvios.tolist())
print((desvios ** 2).tolist())
print(round((desvios ** 2).sum() / (len(variable) - 1), 1))
print(round(variable.var(), 1), round(variable.std(), 1))

[-500.0, -200.0, 200.0, 500.0]
[250000.0, 40000.0, 40000.0, 250000.0]
193333.3
193333.3 439.7


In [11]:
media: float = precio.mean()
desv: float = precio.std()
print(round(media), round(desv))
print(round(media - desv), round(media + desv))

en_banda: pd.Series = (precio - media).abs() <= desv
print(round(en_banda.mean(), 3))

1243 565
678 1808
0.685


In [12]:
print(precio.quantile([0.25, 0.5, 0.75]))
print(precio.quantile(0.9))

q1: float = float(precio.quantile(0.25))
q3: float = float(precio.quantile(0.75))
print(q3 - q1)

0.25     840.0
0.50    1150.0
0.75    1582.5
Name: precio, dtype: float64
1980.0
742.5


In [13]:
print(precio.min(), q1, precio.median(), q3, precio.max())

260 840.0 1150.0 1582.5 3930


In [14]:
print(pisos[["metros", "precio"]].describe().round(1))

       metros  precio
count   600.0   600.0
mean     80.0  1243.0
std      28.6   565.1
min      28.0   260.0
25%      57.8   840.0
50%      81.0  1150.0
75%     104.0  1582.5
max     159.0  3930.0


### Referencia: medidas de dispersión

| Código | Qué da |
|---|---|
| `s.max() - s.min()` | Rango: la distancia entre el mayor y el menor |
| `s.var()` | Varianza: la media de los desvíos al cuadrado (en unidades al cuadrado) |
| `s.std()` | Desviación típica: la raíz de la varianza, en las unidades de los datos |
| `s.std(ddof=0)` | Dividiendo entre n en lugar de n − 1, como `np.std` |
| `s.quantile(0.9)` | Percentil 90: el 90 % de los valores está por debajo |
| `s.quantile([0.25, 0.5, 0.75])` | Los tres cuartiles: Q1, la mediana y Q3 |
| `q3 - q1` | Rango intercuartílico (IQR): dónde está el 50 % central |

**`describe()`:** `count` (valores no vacíos), `mean`, `std`, `min`, `25%` (Q1), `50%` (mediana), `75%` (Q3) y `max`.

**n − 1 o n:** `serie.std()` de pandas divide entre n − 1 (desviación típica de una muestra); `np.std(array)` divide entre n. Con `ddof=0` pandas divide entre n.

In [15]:
print(round(variable.std(), 1))
print(round(np.std(variable.to_numpy()), 1))
print(round(variable.std(ddof=0), 1))

439.7
380.8
380.8


## 4. Forma y valores atípicos

In [16]:
limites: range = range(0, 4500, 500)
tramos: pd.Series = pd.cut(precio, bins=limites)
print(tramos.value_counts(sort=False))

precio
(0, 500]         22
(500, 1000]     210
(1000, 1500]    200
(1500, 2000]    112
(2000, 2500]     36
(2500, 3000]     13
(3000, 3500]      5
(3500, 4000]      2
Name: count, dtype: int64


In [17]:
print(round(metros.mean(), 1), metros.median())
print(round(precio.mean(), 1), precio.median())
print(pisos[["metros", "precio"]].skew().round(2))

80.0 81.0
1243.0 1150.0
metros    0.00
precio    1.07
dtype: float64


In [18]:
iqr: float = q3 - q1
limite_bajo: float = q1 - 1.5 * iqr
limite_alto: float = q3 + 1.5 * iqr
print(limite_bajo, limite_alto)

caros: pd.DataFrame = pisos.loc[precio > limite_alto, columnas]
print(len(caros))
print(caros.sort_values("precio").head(3))

-273.75 2696.25
13
             barrio   tipo  metros  precio
67           Centro  ático     114    2790
447         Levante   piso     111    2790
380  Rincón de Loix  ático     123    2850


In [19]:
pisos["z_precio"] = ((precio - media) / desv).round(2)
print(pisos.loc[0:2, ["precio", "z_precio"]])
print(pisos["z_precio"].max())
print((pisos["z_precio"].abs() > 3).sum())

   precio  z_precio
0    1830      1.04
1     570     -1.19
2    1040     -0.36
4.75
8


In [20]:
def resumen(serie: pd.Series) -> pd.Series:
    return pd.Series({
        "media": serie.mean(),
        "mediana": serie.median(),
        "desv": serie.std(),
        "iqr": serie.quantile(0.75) - serie.quantile(0.25),
        "asimetria": serie.skew(),
    })

tabla: pd.DataFrame = pd.DataFrame(
    {"metros": resumen(metros), "precio": resumen(precio)}
).round(2)
print(tabla)

           metros   precio
media       80.04  1242.97
mediana     81.00  1150.00
desv        28.59   565.12
iqr         46.25   742.50
asimetria    0.00     1.07


### Referencia: forma y valores atípicos

| Código | Qué da |
|---|---|
| `pd.cut(s, bins=range(0, 4500, 500))` | Cada valor, en su tramo: `(500, 1000]` va de más de 500 a 1000 incluido |
| `.value_counts(sort=False)` | Cuántos valores caen en cada tramo, en orden: la tabla de un histograma |
| `s.skew()` | Asimetría: cerca de 0, simétrica; positiva, cola a la derecha; negativa, a la izquierda |
| `q1 - 1.5 * iqr` · `q3 + 1.5 * iqr` | Límites de la regla del IQR: fuera de ellos, valor atípico |
| `(s - s.mean()) / s.std()` | Puntuación z: a cuántas desviaciones típicas de la media está cada valor |

**Cómo describir una variable numérica:** el centro (media y mediana), la dispersión (desviación típica e IQR), la forma (si media y mediana se separan, hay asimetría) y los valores atípicos (regla del IQR).